In [1]:
import openai
import instructor
from pydantic import BaseModel, Field
import os
import json
from typing import Any
from superlinked import framework as sl
from api.agents.superlinked_app.index import business_index, business
from api.agents.superlinked_app.query import query
from api.agents.superlinked_app.utils.utils import *



/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


00:51:39 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1293.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


00:51:39 superlinked.framework.common.space.embedding.model_based.embedding_engine_manager INFO   Consider caching model dimension.
00:51:39 superlinked.framework.dsl.index.index INFO   initialized index


In [2]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: What is your name?
"""

In [3]:
openai_response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": prompt}],
    temperature=0
)
print(openai_response.choices[0].message.content)


I am an AI language model and don't have a personal name, but you can call me Assistant! How can I help you today?


### Add instructor

In [4]:
client = instructor.from_openai(openai.OpenAI())


In [5]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question.")


In [6]:
response, raw_response = client.chat.completions.create_with_completion(
    model="gpt-4o-mini",
    messages=[{"role": "system", "content": prompt}],
    temperature=0,
    response_model=RAGGenerationResponse,
)



In [7]:
response

RAGGenerationResponse(answer='I am an AI assistant and do not have a personal name like a human. You can call me Assistant!')

In [8]:
raw_response

ChatCompletion(id='chatcmpl-DRshrrYvDwf6vGGXKdMAN7BWAWgra', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_fk9qXJ07hsaruMl7vvoJo86x', function=Function(arguments='{"answer":"I am an AI assistant and do not have a personal name like a human. You can call me Assistant!"}', name='RAGGenerationResponse'), type='function')]))], created=1775537519, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_e738e3044b', usage=CompletionUsage(completion_tokens=26, prompt_tokens=95, total_tokens=121, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [9]:
class RAGGenerationResponse(BaseModel):
    answer: str =Field(description="the answer to the question")
    reasoning: str=Field(description="The reasoning for the answer")

In [10]:
response, raw_response= client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system", "content": prompt}
    ],
    temperature=0,
    response_model= RAGGenerationResponse
)

In [14]:
response

RAGGenerationResponse(answer='My name is ChatGPT.', reasoning='I am an AI language model created by OpenAI, and I am commonly known as ChatGPT.')

### RAG Example

In [11]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")

In [12]:
qdrant_vdb = sl.QdrantVectorDatabase(
    url="http://localhost:6333",
    # Superlinked's QdrantVectorDatabase currently requires an api_key arg.
    # For local Qdrant this is typically unused, so we default to empty.
    api_key=os.getenv("QDRANT_API_KEY", ""),
)
parser = sl.DataFrameParser(business)

source_qdrant = sl.RestSource(
    business,
    parser=parser,
)

# RestExecutor needs sl.RestQuery (path for /api/v1/search/<query_path> by default).
business_rest_query = sl.RestQuery(
    rest_descriptor=sl.RestDescriptor(query_path="business_search"),
    query_descriptor=query,
)

executor_qdrant = sl.RestExecutor(
    sources=[source_qdrant],
    indices=[business_index],
    vector_database=qdrant_vdb,
    queries=[business_rest_query],
)
qdrant_app = executor_qdrant.run()


def Retrieve_context(question, qdrant_app, k=5):
    qdrant_results = qdrant_app.query(
    query,
    natural_query=question,
    limit=k,
)

    format_minute_columns_to_hhmm(sl.PandasConverter.to_pandas(qdrant_results))
    return qdrant_results


def _result_to_restaurants(result) -> list[dict[str, Any]]:
    df_columns = ["business_id", "name", "address", "city", "state", "postal_code", "latitude", "longitude", "stars", "review_count", "is_open", "categories", "attributes", "hours"]
    df = sl.PandasConverter.to_pandas(result).rename(columns={"id": "business_id"})
    # Derive `is_open` robustly.
    # Some payloads may contain only `is_open_i` (0/1) instead of the boolean `is_open`.
    if "is_open" not in df.columns:
        if "is_open_i" in df.columns:
            df["is_open"] = df["is_open_i"].astype(int)
        else:
            df["is_open"] = 0

    df = df.assign(
        categories=df.get("category_tags", df.get("categories_text")),
        is_open=df["is_open"].astype(int),
    )

    # Parse attributes/hours when present.
    for c in ("attributes", "hours"):
        if c in df.columns:
            df[c] = df[c].map(
                lambda v: json.loads(v) if isinstance(v, str) and v.strip()
                else ({} if v in ("", None) else v)
            )
        else:
            df[c] = {}
    return df.reindex(columns=df_columns).to_dict(orient="records")


def build_prompt(preprocessed_context, question):
    prompt=f"""
    You are a yelp shopping assistant that can answer question about the restaurants.
    You will be given a question and a list of context.
    Instructions:
    - You need to answer questions based on the provided context only
    - Never use the word context and rfer to it as the available businesses or amenities
    - respond naturally and provide as much details as possible to the user request 
    - Refrain from using filter sush as is_open =True ...Rather say open today

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt


def generate_answer(prompt):
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role":"system", "content": prompt}],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(question):
    context=Retrieve_context(question, qdrant_app)
    preprocessed_context=_result_to_restaurants(context)
    prompt=build_prompt(preprocessed_context, question)
    answer=generate_answer(prompt)

    final_response={
        "datamodel":answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": [e.id for e in context.entries],
        "retrieved_restaurant_names": [e.fields.get("name") for e in context.entries],
        "similarity_score": [e.metadata.score for e in context.entries],
    }
    return final_response

/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:87: UserWarning: Api key is used with an insecure connection.
  self._client = AsyncQdrantClient(
/Users/wilfriedtcheumaha/Code/yelp-assistant/.venv/lib/python3.12/site-packages/superlinked/framework/storage/qdrant/qdrant_vdb_connector.py:92: UserWarning: Api key is used with an insecure connection.
  self._sync_client = QdrantClient(


00:52:15 superlinked.framework.query.query_dag_evaluator INFO   initialized query dag
00:52:15 superlinked.framework.online.online_dag_evaluator INFO   initialized entity dag


In [15]:
output=rag_pipeline("What is the best restaurant in Tampa?")

00:53:49 superlinked.framework.common.delayed_evaluator INFO   Processed sentence-transformers/all-MiniLM-L6-v2 embed
00:53:49 superlinked.framework.query.query_dag_evaluator INFO   evaluated query
00:53:49 superlinked.framework.dsl.executor.query.query_executor INFO   executed query


In [16]:
output

{'datamodel': RAGGenerationResponse(answer='The best restaurant in Tampa based on ratings and reviews from the available options is Shells Seafood Restaurant - South Tampa. It has a high rating of 4.5 stars with 453 reviews. It offers seafood in a casual yet classy atmosphere, has a full bar, is good for kids, and provides services like delivery, takeout, and catering. It is open today from 4 PM to 10 PM on Friday. If you are looking for a highly rated dining experience, Shells Seafood Restaurant would be a great choice.'),
 'answer': 'The best restaurant in Tampa based on ratings and reviews from the available options is Shells Seafood Restaurant - South Tampa. It has a high rating of 4.5 stars with 453 reviews. It offers seafood in a casual yet classy atmosphere, has a full bar, is good for kids, and provides services like delivery, takeout, and catering. It is open today from 4 PM to 10 PM on Friday. If you are looking for a highly rated dining experience, Shells Seafood Restaurant 

In [17]:
print(output["answer"])

The best restaurant in Tampa based on ratings and reviews from the available options is Shells Seafood Restaurant - South Tampa. It has a high rating of 4.5 stars with 453 reviews. It offers seafood in a casual yet classy atmosphere, has a full bar, is good for kids, and provides services like delivery, takeout, and catering. It is open today from 4 PM to 10 PM on Friday. If you are looking for a highly rated dining experience, Shells Seafood Restaurant would be a great choice.
